In [5]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1])) 

from src.data import load_ds

ds = load_ds()
ds

DatasetDict({
    train: Dataset({
        features: ['observation_1', 'observation_2', 'hypothesis_1', 'hypothesis_2', 'label', 'target'],
        num_rows: 16801
    })
    validation: Dataset({
        features: ['observation_1', 'observation_2', 'hypothesis_1', 'hypothesis_2', 'label', 'target'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['observation_1', 'observation_2', 'hypothesis_1', 'hypothesis_2', 'label', 'target'],
        num_rows: 1532
    })
})

In [6]:
{split: len(ds[split]) for split in ds}

{'train': 16801, 'validation': 1000, 'test': 1532}

In [7]:
train_df = ds["train"].to_pandas()
train_df.head()

,observation_1,observation_2,hypothesis_1,hypothesis_2,label,target
0,A baby panda was lost in the woods.,"He was reunited with his parents, who also liv...",He was never lost.,All the people from the zoo looked for him.,2,1
1,A bank teller told me to rob the bank where sh...,We split the money equally.,The mountain told me when no one would be there.,The teller told me when no one would be there.,2,1
2,A bee was buzzing around my head.,I didn't bother to explain the situation.,It was because of the honey in my hair.,It was because of the blood in my stool.,1,0
3,A blizzard suddenly started in the winter.,So the neighbor gave the boy some cash.,A boy offered to shovel his neighbors driveway.,A boy never offered to mow his neighbors drive...,1,0
4,A boy was outside playing when he saw a balloo...,Once he held the balloon he popped it with a s...,The boy popped the balloon.,The boy caught the ballon.,2,1


In [8]:
example = ds["train"][0]
for key, value in example.items():
    print(f"{key:>14}: {value}")

 observation_1: A baby panda was lost in the woods.
 observation_2: He was reunited with his parents, who also lived at the zoo.
  hypothesis_1: He was never lost.
  hypothesis_2: All the people from the zoo looked for him.
         label: 2
        target: 1


In [9]:
import pandas as pd

pd.DataFrame({split: ds[split].to_pandas()["label"].value_counts(normalize=True) for split in ds}).round(3)

,train,validation,test
label,,,
1,0.504,0.508,0.51
2,0.496,0.492,0.49


In [10]:
import re

from src.data import TEXT_COLUMNS

texts = pd.concat([train_df[col] for col in TEXT_COLUMNS], ignore_index=True)
print(f"{len(texts)} text fields")

67204 text fields


In [ ]:
# special character occurrences
patterns = {
    "html tag": r"<[^>]+>",
    "html entity": r"&[a-zA-Z]+;|&#\d+;",
    "url": r"https?://|www\.",
    "non-ascii char": r"[^\x00-\x7f]",
    "broken encoding (U+FFFD)": "\ufffd",
    "curly quote/apostrophe": r"[\u2018\u2019\u201c\u201d]",
    "digit": r"\d",
    "double whitespace": r"\s{2,}",
    "leading/trailing whitespace": r"^\s|\s$",
}
pd.DataFrame(
    {name: [texts.str.contains(pat).sum()] for name, pat in patterns.items()},
    index=["# fields"],
).T.assign(share=lambda d: (d["# fields"] / len(texts)).round(4))

,# fields,share
html tag,0,0.0000
html entity,0,0.0000
url,0,0.0000
non-ascii char,38,0.0006
broken encoding (U+FFFD),3,0.0000
curly quote/apostrophe,32,0.0005
digit,1091,0.0162
double whitespace,68,0.0010
leading/trailing whitespace,0,0.0000


In [12]:
for name, pat in patterns.items():
    hits = texts[texts.str.contains(pat)]
    if len(hits):
        print(f"--- {name} ({len(hits)})")
        for t in hits.head(3):
            print("  ", repr(t))

--- non-ascii char (38)
   "Jo��o was frustrated at his country's government."
   'He opened the envelope and was disappointed he hadn���t been accepted.'
   'Alex took his clothes to his Mom���s house for her to clean.'
--- broken encoding (U+FFFD) (3)
   "Jo��o was frustrated at his country's government."
   'He opened the envelope and was disappointed he hadn���t been accepted.'
   'Alex took his clothes to his Mom���s house for her to clean.'
--- curly quote/apostrophe (32)
   'Iris catch down Lisa’s phone to the floor.".'
   'John’s airline decided to take its workers on a company outing.'
   'All of Judy’s plants were gone.'
--- digit (1091)
   'Anne had just turned 50 when she noticed something unsettling.'
   "Bart's family had not been together for Thanksgiving for 3 years."
   'Bob wanted to do something for his 40th birthday.'
--- double whitespace (68)
   "OUCH!  Mary's foot was hurt, but she kept on playing."
   'Brett was sick but went through   with the speech.'
   'Jaso

In [ ]:
# specail character occurrences
pd.Series([ch for t in texts for ch in t if ord(ch) > 127]).value_counts().head(20)

’    32
�     8
é     2
´     1
Name: count, dtype: int64

In [14]:
# text length distribution
lengths = texts.str.split().str.len()
print("empty fields:", (texts.str.strip() == "").sum())
lengths.describe().round(1)

empty fields: 0


count    67204.0
mean         8.5
std          2.9
min          1.0
25%          6.0
50%          8.0
75%         10.0
max         30.0
dtype: float64

In [ ]:
# pretrained word2vec (first run downloads ~1.6 GB into data/gensim/)
import os

os.environ["GENSIM_DATA_DIR"] = str(Path.cwd().parents[1] / "data" / "gensim")  # set before importing gensim
import gensim.downloader as api

w2v = api.load("word2vec-google-news-300")
print(f"{len(w2v.key_to_index):,} words in word2vec vocab")

In [ ]:
# share of tokens unknown to word2vec, with and without lowercasing
from collections import Counter


def tokenize(text):
    # words (keeping "didn't" together) and single punctuation marks
    return re.findall(r"\w+(?:'\w+)?|[^\w\s]", text)


def unknown_share(counts, vocab):
    unknown = sum(c for word, c in counts.items() if word not in vocab)
    return round(100 * unknown / sum(counts.values()), 2)


counts = Counter(tok for t in texts for tok in tokenize(t))
lower_counts = Counter(tok.lower() for t in texts for tok in tokenize(t))

print(f"unknown tokens, original case: {unknown_share(counts, w2v)} %")
print(f"unknown tokens, lowercased:    {unknown_share(lower_counts, w2v)} %")

In [ ]:
# most frequent unknown tokens
unknown = Counter({word: c for word, c in counts.items() if word not in w2v})
unknown.most_common(20)

In [ ]:
# are stopwords, contractions and punctuation in the word2vec vocab?
for word in ["the", "a", "and", "of", "to", "didn't", "Mom", "mom", "."]:
    print(f"{word:>7}: {word in w2v}")

In [ ]:
# own vocab built on train: share of validation tokens that become <unk>, per min_freq
val_df = ds["validation"].to_pandas()
val_texts = pd.concat([val_df[col] for col in TEXT_COLUMNS])
val_counts = Counter(tok.lower() for t in val_texts for tok in tokenize(t))

for min_freq in [1, 2, 3, 5, 10]:
    vocab = {word for word, c in lower_counts.items() if c >= min_freq}
    print(f"min_freq={min_freq:>2}: vocab size {len(vocab):>6}, unknown in validation {unknown_share(val_counts, vocab)} %")